# AMC — 自动调制识别训练（Kaggle GPU 版）

在 Kaggle 上运行：New Notebook → Settings → Accelerator 选 **GPU T4/P100** →
把下面 `REPO` 改成你的仓库地址 → Run All。

本 notebook 完成：克隆仓库 → 生成 IQ 数据集（5 类调制 × 受损信道链）→
训练 1D-ResNet → 输出混淆矩阵 + 准确率-vs-SNR 曲线。

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/YZml1507/phy-layer-sim-amc.git"
WORKDIR = "/kaggle/working/phy-layer-sim-amc"

if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", REPO, WORKDIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                os.path.join(WORKDIR, "[amc]")], check=True)
sys.path.insert(0, os.path.join(WORKDIR, "src"))

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
import numpy as np
from phylayer.iqdata import CLASSES, generate_dataset

# GPU 上数据可以放开：每类 4000 训练 / 1000 测试
Xtr, ytr, _   = generate_dataset(CLASSES, n_per_class=4000, seed=11)
Xte, yte, ste = generate_dataset(CLASSES, n_per_class=1000, seed=99)
Xtr.shape, Xte.shape

In [ ]:
from phylayer.amc import AMCNet, train_model

model = AMCNet(n_classes=len(CLASSES))
print(f"params: {sum(p.numel() for p in model.parameters()):,}")
losses = train_model(model, Xtr, ytr, epochs=25, batch_size=128)
import matplotlib.pyplot as plt
plt.plot(losses); plt.xlabel('epoch'); plt.ylabel('CE loss'); plt.grid(alpha=.3); plt.show()

In [ ]:
from phylayer.amc import predict, confusion_matrix, accuracy_vs_snr

pred = predict(model, Xte)
acc = (pred == yte).mean()
print(f"test acc (mixed SNR): {acc:.3f}")

cm = confusion_matrix(yte, pred, len(CLASSES))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im = axes[0].imshow(cm / cm.sum(1, keepdims=True), cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(len(CLASSES)), CLASSES)
axes[0].set_yticks(range(len(CLASSES)), CLASSES)
axes[0].set_title(f'confusion (acc={acc:.2f})')
edges = np.arange(-4, 26, 4)
ctr, ac = accuracy_vs_snr(yte, pred, ste, edges)
axes[1].plot(ctr, ac, 'o-'); axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel('SNR (dB)'); axes[1].set_title('accuracy vs SNR')
axes[1].grid(alpha=.3); plt.show()

## 调参方向

- 数据：`n_per_class` 翻倍、`n_samples` 加到 2048（网络感受野更大）
- 网络：`AMCNet(width=96)` 加宽；加深 stages
- 训练：`epochs=40`，`lr` 余弦退火，mixup 增广
- 经典对照：O'Shea RML2016 数据集上 CNN ~85%@高SNR 是常见 baseline